<a href="https://colab.research.google.com/github/srichaithanyareddy/AMCAT-Report/blob/main/portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import datetime
from collections import defaultdict
from mstarpy import Funds

def load_transactions(filename):
    with open("/content/transaction_detail.json", 'r') as file:
        return json.load(file)

def calculate_units(transactions):
    # Dictionary to store units by folio and scheme
    units = defaultdict(lambda: defaultdict(float))
    # Ensure transactions is a list before sorting
    if isinstance(transactions, str):  # Check if transactions is a string (JSON)
        transactions = json.loads(transactions) # if it is a string try loading it
    elif not isinstance(transactions, list):  # Check if it is not a list
        raise TypeError("transactions must be a list or a JSON string")

    # Use a try-except block to handle potential errors during sorting
    try:
        transactions_sorted = sorted(transactions, key=lambda x: x['trxnDate'])
    except TypeError as e:
        print(f"Error sorting transactions: {e}")  # Print the error
        print("Transactions data:", transactions)  # Print the transactions data for debugging
        raise  # Re-raise the exception to stop execution

    for txn in transactions_sorted:
        folio = txn['folio']
        scheme = txn['isin']
        units_amount = float(txn['trxnUnits'])
        purchase_price = float(txn['purchasePrice'])

        if units_amount > 0:  # Purchase
            units[folio][scheme] += units_amount
        else:  # Sale (negative units)
            units[folio][scheme] += units_amount  # This will reduce the count

    return units

def fetch_current_nav(isin):
    fund = Funds(term=isin, country="in")
    end_date = datetime.datetime.now()
    start_date = end_date - datetime.timedelta(days=365)
    history = fund.nav(start_date=start_date, end_date=end_date, frequency="daily")
    return history[-1]['nav']  # Last day's NAV

def calculate_portfolio_value(units):
    total_value = 0
    portfolio_details = {}

    for folio, schemes in units.items():
        for scheme, amount in schemes.items():
            if amount > 0:  # Only consider holdings
                nav = fetch_current_nav(scheme)
                current_value = amount * nav
                # Check if transactions is not empty and has 'purchasePrice'
                if transactions and 'purchasePrice' in transactions[-1]:
                    acquisition_cost = amount * float(transactions[-1]['purchasePrice'])  # Last purchase price
                else:
                    acquisition_cost = 0  # Set to 0 if no purchasePrice available
                gain = current_value - acquisition_cost

                portfolio_details[scheme] = {
                    'units': amount,
                    'current_value': current_value,
                    'gain': gain,
                    'NAV': nav
                }
                total_value += current_value

    return portfolio_details, total_value

def main():
    # Load transactions
    transactions = load_transactions('/content/transaction_detail.json')

    # Calculate units based on transactions
    units = calculate_units(transactions)

    # Calculate portfolio value and details
    portfolio_details, total_value = calculate_portfolio_value(units)

    # Display the results
    print("Portfolio Details:")
    for scheme, details in portfolio_details.items():
        print(f"Scheme: {scheme}, Units: {details['units']:.2f}, Current Value: ₹{details['current_value']:.2f}, Gain: ₹{details['gain']:.2f}, NAV: ₹{details['NAV']:.2f}")

In [ ]:
pip install mstarpy